# KvForge ProLAD — Long Prompt Benchmark v2


In [ ]:
import json, math, time, gc
import torch
import torch.nn as nn
import torch.nn.functional as F

print("=" * 70)
print("KvForge — ProLAD Long Prompt Benchmark v2")
print("=" * 70)

device = "cuda" if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 7 else "cpu"
print(f"Device: {device}")

from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.cache_utils import DynamicCache

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
dtype = torch.float16 if device == "cuda" else torch.float32

tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=dtype).to(device).eval()

# LoRA
class LoRALinear(nn.Module):
    def __init__(self, orig, r=8, alpha=16):
        super().__init__()
        self.orig = orig; self.scaling = alpha / r
        dt = orig.weight.dtype
        self.lora_A = nn.Parameter(torch.randn(orig.in_features, r, dtype=dt) * 0.02)
        self.lora_B = nn.Parameter(torch.zeros(r, orig.out_features, dtype=dt))
        self.active = True
    def activate(self, a=True): self.active = a
    def forward(self, x):
        h = self.orig(x)
        if self.active: h = h + (x @ self.lora_A @ self.lora_B) * self.scaling
        return h

def inject_lora(model, r=8):
    count = 0
    for n, m in model.named_modules():
        if any(n.endswith(s) for s in [".q_proj", ".k_proj", ".v_proj", ".o_proj"]):
            if isinstance(m, nn.Linear) and not isinstance(m, LoRALinear):
                parent = model; parts = n.split(".")
                for p in parts[:-1]:
                    if p: parent = getattr(parent, p)
                setattr(parent, parts[-1], LoRALinear(m, r=r))
                count += 1
    return count

def set_lora(m, a):
    for mod in m.modules():
        if hasattr(mod, "activate"): mod.activate(a)

n_lora = inject_lora(base, r=8)
lora_p = sum(p.numel() for n,p in base.named_parameters() if "lora" in n)
print(f"LoRA: {n_lora} modules, {lora_p/1e3:.1f}K params")

# ProLAD schedule
def prolad_activate(model, step, total, sched="cosine"):
    modules = [(n,m) for n,m in model.named_modules() if hasattr(m,"activate") and hasattr(m,"lora_A")]
    n_tot = len(modules)
    progress = step / max(total - 1, 1)
    if sched == "immediate": n_act = n_tot
    elif sched == "cosine": n_act = max(1, int(n_tot * (1 - math.cos(progress * math.pi / 2))))
    else: n_act = n_tot
    for i, (_, mod) in enumerate(modules):
        mod.activate(i < n_act)
    return n_act, n_tot

texts = [
    "The transformer architecture uses self-attention to process sequences in parallel.",
    "KV cache compression reduces memory by quantizing key-value pairs during inference.",
    "Language models generate text by autoregressively predicting the next token.",
]

def train_model(model, steps=80, sched="cosine"):
    opt = torch.optim.AdamW([p for n,p in model.named_parameters() if "lora" in n], lr=3e-3)
    model.train()
    losses = []
    for s in range(steps):
        ids = tok(texts[s % 3], return_tensors="pt", truncation=True, max_length=128).to(device)["input_ids"]
        prolad_activate(model, s, steps, sched)
        loss = F.cross_entropy(model(ids).logits[0, :-1], ids[0, 1:])
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    model.eval()
    return losses

# Generate natural long text
wiki_text = (
    "The transformer is a deep learning architecture that relies on the parallel multi-head attention mechanism. "
    "Unlike recurrent neural networks, transformers process all tokens simultaneously. "
    "The self-attention mechanism computes weights that represent the relevance of each token to every other token. "
    "These weights are calculated using query, key, and value matrices derived from the input embeddings. "
    "Multi-head attention runs multiple attention operations in parallel, each with different learned projections. "
    "The outputs are concatenated and linearly transformed. "
    "Positional encodings are added to the input embeddings to capture token order information. "
    "Layer normalization is applied before each sub-layer, and residual connections bypass each sub-layer. "
    "The feed-forward network consists of two linear transformations with a ReLU activation in between. "
    "Transformers have become the foundation of modern natural language processing. "
    "Models like BERT, GPT, T5, and Llama are all based on the transformer architecture. "
    "The key advantage of transformers is their ability to capture long-range dependencies without the vanishing gradient problem. "
    "Training transformers requires large amounts of data and computational resources. "
    "Despite their size, transformers achieve state-of-the-art results on virtually every NLP benchmark. "
) * 4  # ~1200 tokens

short_prompt = "The transformer architecture uses self-attention to process sequences."

# Measure PPL gap function
def measure_gap(model, inp):
    ids = inp["input_ids"]
    with torch.no_grad():
        set_lora(model, True)
        ppl_on = math.exp(F.cross_entropy(model(ids).logits[0, :-1], ids[0, 1:]).item())
        set_lora(model, False)
        ppl_off = math.exp(F.cross_entropy(model(ids).logits[0, :-1], ids[0, 1:]).item())
    return ppl_on, ppl_off

# ==== Phase 1: Train ProLAD ====
print("\\nPhase 1: ProLAD training (cosine schedule)...")
l1 = train_model(base, 80, "cosine")
print(f"  {l1[0]:.4f} -> {l1[-1]:.4f}")

# Save ProLAD weights as a DICT
prolad_state = {}
for n, p in base.named_parameters():
    if "lora" in n:
        prolad_state[n] = p.data.clone()

# Measure ProLAD gaps
inp_short = tok(short_prompt, return_tensors="pt", truncation=True, max_length=128).to(device)
inp_long = tok(wiki_text, return_tensors="pt", truncation=True, max_length=2048).to(device)

print(f"\\nShort prompt: {inp_short['input_ids'].shape[1]} tokens")
print(f"Long prompt:  {inp_long['input_ids'].shape[1]} tokens")

p_on_s, p_off_s = measure_gap(base, inp_short)
p_on_l, p_off_l = measure_gap(base, inp_long)
gap_s = abs(p_on_s - p_off_s)
gap_l = abs(p_on_l - p_off_l)

print(f"\\n  ProLAD SHORT: PPL(on)={p_on_s:.2f} PPL(off)={p_off_s:.2f} Gap={gap_s:.4f}")
print(f"  ProLAD LONG:  PPL(on)={p_on_l:.2f} PPL(off)={p_off_l:.2f} Gap={gap_l:.4f}")

# ==== Phase 2: Train Baseline (immediate) ====
print("\\nPhase 2: Baseline training (immediate schedule)...")

# RESET LoRA weights to random
for n, p in base.named_parameters():
    if "lora_A" in n: nn.init.normal_(p, 0, 0.02)
    elif "lora_B" in n: nn.init.zeros_(p)

l2 = train_model(base, 80, "immediate")
print(f"  {l2[0]:.4f} -> {l2[-1]:.4f}")

# Measure Baseline gaps
b_on_s, b_off_s = measure_gap(base, inp_short)
b_on_l, b_off_l = measure_gap(base, inp_long)
b_gap_s = abs(b_on_s - b_off_s)
b_gap_l = abs(b_on_l - b_off_l)

print(f"\\n  Baseline SHORT: PPL(on)={b_on_s:.2f} PPL(off)={b_off_s:.2f} Gap={b_gap_s:.4f}")
print(f"  Baseline LONG:  PPL(on)={b_on_l:.2f} PPL(off)={b_off_l:.2f} Gap={b_gap_l:.4f}")

# ==== Phase 3: Restore ProLAD + measure again ====
print("\\nPhase 3: Restore ProLAD weights + re-measure...")
for n, p in base.named_parameters():
    if n in prolad_state:
        p.data.copy_(prolad_state[n])

p_on_s2, p_off_s2 = measure_gap(base, inp_short)
p_on_l2, p_off_l2 = measure_gap(base, inp_long)
gap_s2 = abs(p_on_s2 - p_off_s2)
gap_l2 = abs(p_on_l2 - p_off_l2)

print(f"  ProLAD SHORT: PPL(on)={p_on_s2:.2f} PPL(off)={p_off_s2:.2f} Gap={gap_s2:.4f}")
print(f"  ProLAD LONG:  PPL(on)={p_on_l2:.2f} PPL(off)={p_off_l2:.2f} Gap={gap_l2:.4f}")

# ==== Summary ====
print("\\n" + "=" * 70)
print("FINAL RESULTS")
print("=" * 70)
print(f"  {'Method':<15} {'Prompt':<8} {'PPL(on)':>10} {'PPL(off)':>10} {'Gap':>10}")
print(f"  {'-'*15} {'-'*8} {'-'*10} {'-'*10} {'-'*10}")
print(f"  {'ProLAD':<15} {'SHORT':<8} {p_on_s:>10.2f} {p_off_s:>10.2f} {gap_s:>10.4f}")
print(f"  {'Baseline':<15} {'SHORT':<8} {b_on_s:>10.2f} {b_off_s:>10.2f} {b_gap_s:>10.4f}")
print(f"  {'ProLAD':<15} {'LONG':<8} {p_on_l2:>10.2f} {p_off_l2:>10.2f} {gap_l2:>10.4f}")
print(f"  {'Baseline':<15} {'LONG':<8} {b_on_l:>10.2f} {b_off_l:>10.2f} {b_gap_l:>10.4f}")

# Analysis
imp_s = (b_gap_s - gap_s) / max(b_gap_s, 0.001) * 100
imp_l = (b_gap_l - gap_l2) / max(b_gap_l, 0.001) * 100
print(f"\\n  SHORT gap improvement: {imp_s:.1f}%")
print(f"  LONG gap improvement:  {imp_l:.1f}%")

if gap_l2 < b_gap_l * 1.2:
    print(f"  \\n  FINDING: ProLAD long-prompt gap similar to short-prompt gap.")
    print(f"  → ProLAD reduces long-prompt quality degradation. ✅")
else:
    print(f"  \\n  FINDING: Long prompt gap is larger than short prompt gap.")
    print(f"  → Quality degrades with length — needs more investigation. ⚠️")

# Long vs short gap ratio
print(f"\\n  Long/Short gap ratio:")
print(f"    ProLAD:  {gap_l2/gap_s:.1f}x")
print(f"    Baseline: {b_gap_l/b_gap_s:.1f}x")
print(f"    {'Better' if gap_l2/gap_s < b_gap_l/b_gap_s else 'Worse'} for long prompts")

with open("/kaggle/working/results.json", "w") as f:
    json.dump({
        "prolad": {"short_gap": gap_s, "long_gap": gap_l2, "short_gap_v2": gap_s2},
        "baseline": {"short_gap": b_gap_s, "long_gap": b_gap_l},
    }, f, indent=2)
print("\\nDone! results.json saved.")
